In [15]:
import numpy as np
import matplotlib
matplotlib.use('TkAgg')  # or 'Qt5Agg' — needed for interactive picking outside JupyterLab
import matplotlib.pyplot as plt
import sys
sys.path.append('../results')

from dalean_stable_amplifying_2Dnets import create_miniSOCs
from utils import init_cond, J_dynamics

In [16]:
# Cell 2 — generate solutions (parameters as used in the paper)
solutions = create_miniSOCs(max_wt=5, seed=0, max_solns=1000)
print(f"Found {len(solutions)} valid matrices out of 1k candidates")

Found 333 valid matrices out of 1k candidates


In [17]:
N = 2
n = len(solutions)

l1s = np.zeros(n, dtype=complex)
l2s = np.zeros(n, dtype=complex)
ampl_init    = np.zeros((n, 2))
non_ampl_init = np.zeros((n, 2))
proxy_ampls  = np.zeros(n)

for i, row in enumerate(solutions):
    J = row.reshape(2, 2)
    
    # Eigenvalues
    evals = np.linalg.eigvals(J)
    l1s[i], l2s[i] = evals[0], evals[1]
    
    # Initial conditions
    a = init_cond(J)
    ampl_init[i]     = np.real(a[:, 0])
    non_ampl_init[i] = np.real(a[:, 1])
    
    # Amplification
    _, proxy_ampls[i], _, _ = J_dynamics(J, np.real(a[:, 0]))

# Sort for amplification panel
sort_idx         = np.argsort(proxy_ampls)
proxy_ampl_sorted = proxy_ampls[sort_idx]
count = n

In [35]:
# Interactive figure

artis = []

def clear_highlights():
    for artist in artis:
        try:
            artist.remove()
        except ValueError:
            pass
    artis.clear()

def onpick(event):
    clear_highlights()
    idx = event.ind[0]

    artis.append(ax1.scatter(solutions[idx, 0], solutions[idx, 3],
                             marker='o', color='y', zorder=5))
    artis.append(ax2.scatter(solutions[idx, 1], solutions[idx, 2],
                             marker='o', color='y', zorder=5))
    artis.append(ax3.scatter(np.real(l1s[idx]), np.imag(l1s[idx]),
                             marker='o', color='y', zorder=5))
    artis.append(ax3.scatter(np.real(l2s[idx]), np.imag(l2s[idx]),
                             marker='o', color='y', zorder=5))
    artis.append(ax4.scatter(ampl_init[idx, 0], ampl_init[idx, 1],
                             marker='o', color='y', zorder=5))
    artis.append(ax4.scatter(non_ampl_init[idx, 0], non_ampl_init[idx, 1],
                             marker='o', color='y', zorder=5))
    # Find where this matrix sits in the sorted amplification panel
    sorted_pos = np.where(sort_idx == idx)[0][0]
    artis.append(ax5.scatter(sorted_pos, proxy_ampls[idx],
                             marker='o', color='y', zorder=5))

    fig.canvas.draw()

fig = plt.figure(figsize=(18, 4))
fig.subplots_adjust(wspace=0.4, left=0.05, right=0.97)

ax1 = plt.subplot2grid((1, 5), (0, 0))
ax1.scatter(solutions[:, 0], solutions[:, 3], marker='.', color='tab:blue', picker=4, s=2)
ax1.set_xlabel('a')
ax1.set_ylabel('d')

ax2 = plt.subplot2grid((1, 5), (0, 1))
ax2.scatter(solutions[:, 1], solutions[:, 2], marker='.', color='tab:blue', picker=4, s=2)
ax2.set_xlabel('b')
ax2.set_ylabel('c')

ax3 = plt.subplot2grid((1, 5), (0, 2))
ax3.scatter(np.real(l1s), np.imag(l1s), marker='.', color='tab:gray', picker=4, s=2)
ax3.scatter(np.real(l2s), np.imag(l2s), marker='.', color='tab:gray', picker=4, s=2)
ax3.axvline(0, color='k', linewidth=0.8, linestyle='--')
ax3.set_xlabel(r'$\lambda$ (Real)')
ax3.set_ylabel(r'$\lambda$ (Im)')
ax3.yaxis.set_label_coords(-0.15, 0.5)

ax4 = plt.subplot2grid((1, 5), (0, 3))
ax4.scatter(ampl_init[:, 0], ampl_init[:, 1],
            color='tab:red', marker='.', picker=4, s=2, label='amplifying')
ax4.scatter(non_ampl_init[:, 0], non_ampl_init[:, 1],
            color='k', marker='.', picker=4, s=2, label='non-amplifying')
ax4.plot(0, 0, marker='x', color='blue')
ax4.set_xlabel(r'$n^E_0$')
ax4.set_ylabel(r'$n^I_0$')
ax4.yaxis.set_label_coords(-0.22, 0.5)
ax4.legend(fontsize=7, markerscale=3)

ax5 = plt.subplot2grid((1, 5), (0, 4))
ax5.scatter(np.arange(count), proxy_ampl_sorted[::-1], marker='.', picker=4, s=2)
ax5.set_xlabel('matrices')
ax5.set_ylabel('Amplification')

fig.canvas.mpl_connect('pick_event', onpick)
plt.show()